# ClosetGraph Demo

Notebook demo covering the walking skeleton (Stages 1-2) and weather-aware recommendations (Stage 3).

Build the retrieval index once before running: `python -m closetgraph.indexing`.

In [ ]:
import json
from pathlib import Path

from dotenv import load_dotenv

from closetgraph.graph import build_graph
from closetgraph.models import WardrobeItem

load_dotenv()


def print_ranked_outfits(result, wardrobe):
    item_lookup = {item.id: item.name for item in wardrobe}
    for outfit in result["ranked_outfits"][:5]:
        names = ", ".join(item_lookup[i] for i in outfit.items)
        print(f"{outfit.rank}. {names}")
        print(f"   {outfit.reasoning}")

In [ ]:
wardrobe_path = Path("docs/wardrobe.json")
wardrobe_data = json.loads(wardrobe_path.read_text())
wardrobe = [WardrobeItem(**item) for item in wardrobe_data]

for item in wardrobe:
    print(f"{item.id}: {item.name} ({item.category})")

In [ ]:
graph = build_graph()

In [ ]:
occasion = "first client meeting, conservative office"
print(f"Occasion: {occasion}")
result = graph.invoke(
    {
        "wardrobe": wardrobe,
        "occasion": occasion,
        "candidate_outfits": [],
        "ranked_outfits": [],
    }
)
print_ranked_outfits(result, wardrobe)
print("Retrieved guidance:")
print(result["retrieved_guidance"])

In [ ]:
occasion = "team happy hour, casual bar"
print(f"Occasion: {occasion}")
result = graph.invoke(
    {
        "wardrobe": wardrobe,
        "occasion": occasion,
        "candidate_outfits": [],
        "ranked_outfits": [],
    }
)
print_ranked_outfits(result, wardrobe)
print("Retrieved guidance:")
print(result["retrieved_guidance"])

In [ ]:
occasion = "dinner date, upscale restaurant"
print(f"Occasion: {occasion}")
result = graph.invoke(
    {
        "wardrobe": wardrobe,
        "occasion": occasion,
        "candidate_outfits": [],
        "ranked_outfits": [],
    }
)
print_ranked_outfits(result, wardrobe)
print("Retrieved guidance:")
print(result["retrieved_guidance"])

In [ ]:
occasion = "casual weekend brunch with friends"
print(f"Occasion: {occasion}")
result = graph.invoke(
    {
        "wardrobe": wardrobe,
        "occasion": occasion,
        "candidate_outfits": [],
        "ranked_outfits": [],
    }
)
print_ranked_outfits(result, wardrobe)
print("Retrieved guidance:")
print(result["retrieved_guidance"])

## Stage 3: Weather-aware recommendations

Weather is optional and influences the ranking without overriding the occasion. Provide an explicit `temperature_c`/`condition`, or a `location` to resolve a live forecast from Open-Meteo (keyless). User-provided fields take precedence; a lookup fills only what is missing.

In [ ]:
from closetgraph.models import WeatherInput


def recommend(occasion, weather_input=None):
    result = graph.invoke(
        {
            "wardrobe": wardrobe,
            "occasion": occasion,
            "candidate_outfits": [],
            "ranked_outfits": [],
            "weather_input": weather_input,
        }
    )
    print(f"Occasion: {occasion}")
    print(f"Weather context: {result.get('weather_context')}")
    if result.get("weather_error"):
        print(f"Weather error: {result['weather_error']}")
    print_ranked_outfits(result, wardrobe)
    print()
    return result


# Same occasion, cold vs warm: watch the outer layer move in and out of the ranking.
recommend("weekend walk around the city", WeatherInput(temperature_c=2, condition="cold"))
recommend("weekend walk around the city", WeatherInput(temperature_c=27, condition="sunny"))

In [ ]:
# Real weather: resolve a live forecast from a city name (Open-Meteo, keyless).
# Change the city to your own; a failed lookup degrades gracefully (no fabricated weather).
recommend("outdoor afternoon market", WeatherInput(location="London"))

In [ ]:
from closetgraph.retrieval import retrieve_style_guidance
for occ in ["graphic tee with blazer for a work event",
            "oversized tee layered for client meeting"]:
    print(occ)
    print(retrieve_style_guidance(occ))
    print()

In [ ]:
from closetgraph.stylist import rank_outfits
from closetgraph.candidates import generate_candidates
from closetgraph.retrieval import retrieve_style_guidance

occasion = "black-tie gala"
candidates = generate_candidates(wardrobe)
guidance = retrieve_style_guidance(occasion)
result = rank_outfits(candidates, wardrobe, occasion, guidance)
for r in result:
    print(r.reasoning)

In [ ]:
result[0].items

## Manual Notes

- Does the ranking shift meaningfully between occasions?
- Does the reasoning reference real layering logic or generic filler?
- Are all recommended items present in the wardrobe?